# KAIL — Master Notebook
> **Kora AI Lab** | H-AZR Research Pipeline
>
> **Before running:** `Runtime → Change runtime type → T4 GPU → Save`

---

## Structure

| Section | What it does | Run for scenario |
|---------|-------------|------------------|
| **0 — Setup** | Install deps, login, mount Drive | Always |
| **1 — Baseline eval** | Measure hallucination + code accuracy | Scenario A |
| **2 — H-Neuron probe** | Identify H-Neurons in base model | All |
| **3 — SPIN warmup** | Stage 1: self-play fine-tuning | Scenarios B, C |
| **4 — H-AZR training** | Stage 3: GRPO + H-Neuron penalty | Scenarios C, D |
| **5 — Final eval + Table** | Run all 4 scenarios, produce Table 1 | Paper |

**To resume after a Colab disconnect:** re-run section 0 then jump to where you stopped.

---
# Section 0 — Setup

In [ ]:
# 0.1 — Check GPU
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU:   {torch.cuda.get_device_name(0)}")
print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"torch: {torch.__version__}")

In [ ]:
# 0.2 — Install dependencies (does NOT touch torch/numpy/pandas)
!pip install -q \
  transformers==4.44.0 \
  trl==0.9.6 \
  peft==0.12.0 \
  bitsandbytes==0.43.3 \
  accelerate==0.33.0 \
  datasets \
  evaluate \
  sentencepiece \
  wandb
print("Done. Ignore warnings above.")

In [ ]:
# 0.3 — RESTART RUNTIME NOW
# Runtime → Restart session
# After restart: skip 0.1 and 0.2, run from 0.4 onwards
print(">>> Runtime → Restart session — then continue from cell 0.4 <<<")

In [ ]:
# 0.4 — Imports (run after restart)
import os, json, subprocess, tempfile, gc
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(f"transformers: {__import__('transformers').__version__}")
print(f"trl:          {__import__('trl').__version__}")
print(f"peft:         {__import__('peft').__version__}")
print(f"GPU:          {torch.cuda.get_device_name(0)}")
print("All good.")

In [ ]:
# 0.5 — Config (edit these)
MODEL_NAME     = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
HF_TOKEN       = ""   # huggingface.co/settings/tokens
SCENARIO       = "C"  # "A" | "B" | "C" | "D"
LAMBDA_H       = 0.1  # H-Neuron penalty weight
SPIN_ITERS     = 3
SPIN_SAMPLES   = 128  # lower = faster on free Colab
H_NEURONS_FILE = "/content/drive/MyDrive/KAIL/h_neurons.json"  # from a previous run

# Paths
DRIVE_BASE  = "/content/drive/MyDrive/KAIL"
CKPT_SPIN   = f"{DRIVE_BASE}/checkpoints/spin_final"
CKPT_HAZR   = f"{DRIVE_BASE}/checkpoints/hazr_scenario_{SCENARIO}"
RESULTS_DIR = f"{DRIVE_BASE}/results"

print(f"Scenario: {SCENARIO} | lambda_h: {LAMBDA_H}")

In [ ]:
# 0.6 — HuggingFace login + mount Drive
from huggingface_hub import login, whoami
from google.colab import drive

login(token=HF_TOKEN) if HF_TOKEN else login()
print(f"HF: {whoami()['name']}")

drive.mount('/content/drive')
for d in [DRIVE_BASE, RESULTS_DIR, os.path.dirname(CKPT_SPIN)]:
    os.makedirs(d, exist_ok=True)
print(f"Drive mounted. Base: {DRIVE_BASE}")

In [ ]:
# 0.7 — Model loader (shared by all sections)
def load_model(checkpoint=None):
    """Load base model or a checkpoint in 4-bit QLoRA."""
    source = checkpoint or MODEL_NAME
    print(f"Loading: {source}")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    mdl = AutoModelForCausalLM.from_pretrained(
        source, quantization_config=bnb,
        device_map="auto", trust_remote_code=True
    )
    mdl.eval()
    print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    return mdl, tok

def generate(model, tok, prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

print("Helpers ready.")

---
# Section 1 — Baseline Evaluation (Scenario A)

In [ ]:
# 1.1 — Load model for eval
model, tok = load_model()

In [ ]:
# 1.2 — Code execution helper
def run_code(code, test, timeout=5):
    full = code + "\n\n" + test
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(full); tmp = f.name
    try:
        return subprocess.run(["python", tmp], capture_output=True, timeout=timeout).returncode == 0
    except: return False
    finally: os.unlink(tmp)

CODE_TASKS = [
    {"p": "Write a Python function `add(a,b)` returning the sum.",
     "t": "assert add(2,3)==5\nassert add(-1,1)==0"},
    {"p": "Write a Python function `is_palindrome(s)` returning True if palindrome.",
     "t": "assert is_palindrome('racecar')\nassert not is_palindrome('hello')"},
    {"p": "Write a Python function `fibonacci(n)` returning nth Fibonacci (0-indexed).",
     "t": "assert fibonacci(0)==0\nassert fibonacci(6)==8"},
    {"p": "Write a Python function `is_prime(n)` returning True if n is prime.",
     "t": "assert is_prime(7)\nassert not is_prime(4)"},
    {"p": "Write a Python function `flatten(lst)` flattening a list of lists.",
     "t": "assert flatten([[1,2],[3,4]])==[1,2,3,4]"},
    {"p": "Write a Python function `count_vowels(s)` returning vowel count.",
     "t": "assert count_vowels('hello')==2\nassert count_vowels('rhythm')==0"},
    {"p": "Write a Python function `binary_search(lst, target)` returning index or -1.",
     "t": "assert binary_search([1,3,5,7],5)==2\nassert binary_search([1,3,5],4)==-1"},
    {"p": "Write a Python function `gcd(a,b)` returning the GCD.",
     "t": "assert gcd(12,8)==4\nassert gcd(7,3)==1"},
]
print(f"{len(CODE_TASKS)} code tasks ready.")

In [ ]:
# 1.3 — Eval function (reused for all scenarios)
def eval_model(model, tok, label=""):
    print(f"\nEvaluating: {label}")

    # TruthfulQA
    ds = load_dataset("truthful_qa", "generation", split="validation")
    correct = 0
    for s in tqdm(list(ds.select(range(80))), desc="TruthfulQA"):
        pred = generate(model, tok, s["question"], max_new_tokens=100)
        correct += int(s["best_answer"].lower()[:25] in pred.lower())
    tqa_acc = correct / 80

    # Code eval
    passed = 0
    for task in tqdm(CODE_TASKS, desc="Code"):
        code = generate(model, tok, task["p"], max_new_tokens=300)
        if "```python" in code: code = code.split("```python")[1].split("```")[0]
        elif "```" in code: code = code.split("```")[1].split("```")[0]
        passed += int(run_code(code, task["t"]))
    code_rate = passed / len(CODE_TASKS)

    result = {
        "label": label,
        "truthfulqa_accuracy": round(tqa_acc, 3),
        "hallucination_rate": round(1 - tqa_acc, 3),
        "code_pass_rate": round(code_rate, 3),
    }
    print(f"  TruthfulQA accuracy: {tqa_acc:.3f}")
    print(f"  Hallucination rate:  {1-tqa_acc:.3f}")
    print(f"  Code pass rate:      {code_rate:.3f}")
    return result

results = {}
results["A"] = eval_model(model, tok, label="A — Baseline")

with open(f"{RESULTS_DIR}/scenario_A.json", "w") as f:
    json.dump(results["A"], f, indent=2)
print("Saved.")

---
# Section 2 — H-Neuron Probe

In [ ]:
# 2.1 — Collect MLP activations
def get_mlp_activations(model, tok, prompt):
    acts = {}
    hooks = []
    for i, layer in enumerate(model.model.layers):
        def hook(m, inp, out, idx=i): acts[idx] = out.detach().cpu().float()
        hooks.append(layer.mlp.register_forward_hook(hook))
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad(): model(**inp)
    for h in hooks: h.remove()
    return {k: v[0, -1, :].numpy() for k, v in acts.items()}

print("Activation collector ready.")

In [ ]:
# 2.2 — Run probe on TriviaQA
# Skip this cell if h_neurons.json already exists on Drive
if os.path.exists(H_NEURONS_FILE):
    print(f"H-Neurons already computed. Loading from {H_NEURONS_FILE}")
    with open(H_NEURONS_FILE) as f:
        h_neuron_data = json.load(f)
    h_neurons = h_neuron_data["layers"]
    print(f"Loaded: {h_neuron_data['total_h_neurons']} H-Neurons ({h_neuron_data['h_neuron_pct']:.3f}%)")
else:
    ds = load_dataset("trivia_qa", "rc.nocontext", split="validation")
    hall_acts, corr_acts = {}, {}
    n_hall = n_corr = 0

    for s in tqdm(list(ds.select(range(200))), desc="Probing"):
        q = s["question"]
        answers = s["answer"]["aliases"]
        pred = generate(model, tok, q, max_new_tokens=60)
        is_hall = not any(a.lower() in pred.lower() for a in answers)
        acts = get_mlp_activations(model, tok, q)
        target = hall_acts if is_hall else corr_acts
        for k, v in acts.items():
            target.setdefault(k, []).append(v)
        if is_hall: n_hall += 1
        else: n_corr += 1

    print(f"Hallucinated: {n_hall} | Correct: {n_corr}")

    h_neurons = {}
    total_n = total_h = 0
    for k in hall_acts:
        if k not in corr_acts: continue
        diff = np.mean(hall_acts[k], axis=0) - np.mean(corr_acts[k], axis=0)
        std = np.std(diff)
        if std < 1e-8: continue
        z = diff / std
        h_idx = np.where(np.abs(z) > 2.0)[0].tolist()
        total_n += len(diff); total_h += len(h_idx)
        if h_idx: h_neurons[str(k)] = {"indices": h_idx, "z_scores": np.abs(z[h_idx]).tolist()}

    pct = 100 * total_h / total_n
    h_neuron_data = {"model": MODEL_NAME, "threshold": 2.0,
                     "n_hall_samples": n_hall, "n_corr_samples": n_corr,
                     "total_neurons": total_n, "total_h_neurons": total_h,
                     "h_neuron_pct": pct, "layers": h_neurons}

    with open(H_NEURONS_FILE, "w") as f:
        json.dump(h_neuron_data, f, indent=2)
    print(f"H-Neurons: {total_h:,} / {total_n:,} ({pct:.3f}%) — saved to Drive")

In [ ]:
# 2.3 — Visualize H-Neuron distribution
layers_idx = sorted([int(k) for k in h_neurons.keys()])
counts = [len(h_neurons[str(l)]["indices"]) for l in layers_idx]
plt.figure(figsize=(12, 3))
plt.bar(layers_idx, counts, color="#1D9E75", alpha=0.85)
plt.xlabel("Layer"); plt.ylabel("H-Neurons")
plt.title(f"H-Neurons per layer — {h_neuron_data['h_neuron_pct']:.3f}% of all neurons")
plt.tight_layout(); plt.show()

---
# Section 3 — SPIN Warmup (Scenarios B and C)

In [ ]:
# 3.1 — Skip this section if running Scenario A or D
RUN_SPIN = SCENARIO in ["B", "C"]
print(f"Run SPIN: {RUN_SPIN} (Scenario {SCENARIO})")

In [ ]:
# 3.2 — SPIN training
if RUN_SPIN:
    from trl import DPOConfig, DPOTrainer

    # Free model memory and reload for training
    del model; gc.collect(); torch.cuda.empty_cache()

    model, tok = load_model()
    model.train()
    model = prepare_model_for_kbit_training(model)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type="CAUSAL_LM",
                      target_modules=["q_proj","k_proj","v_proj","o_proj",
                                      "gate_proj","up_proj","down_proj"])
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    PROMPTS = [
        "Write a Python function that reverses a string.",
        "Write a Python function that checks if a number is prime.",
        "Write a Python function that computes factorial recursively.",
        "Write a Python function that finds the max element in a list.",
        "Write a Python function that flattens a nested list.",
        "Write a Python function that implements binary search.",
        "Write a Python function that removes duplicates from a list.",
        "Write a Python generator that yields Fibonacci numbers.",
    ]

    prev_responses = None
    expanded = (PROMPTS * (SPIN_SAMPLES // len(PROMPTS) + 1))[:SPIN_SAMPLES]

    def gen_responses(prompts):
        out = []
        for p in tqdm(prompts, desc="Generating"):
            msgs = [{"role": "user", "content": p}]
            t = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tok(t, return_tensors="pt", truncation=True, max_length=256).to(model.device)
            with torch.no_grad():
                o = model.generate(**inp, max_new_tokens=300, do_sample=True,
                                   temperature=0.8, pad_token_id=tok.eos_token_id)
            out.append(tok.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True))
        return out

    for it in range(1, SPIN_ITERS + 1):
        print(f"\nSPIN iteration {it}/{SPIN_ITERS}")
        chosen = gen_responses(expanded)
        rejected = prev_responses if prev_responses else gen_responses(expanded)
        prev_responses = chosen

        ds = Dataset.from_dict({"prompt": expanded, "chosen": chosen, "rejected": rejected})

        cfg = DPOConfig(
            output_dir=f"{DRIVE_BASE}/checkpoints/spin_iter_{it}",
            num_train_epochs=1, per_device_train_batch_size=1,
            gradient_accumulation_steps=8, learning_rate=5e-5,
            warmup_ratio=0.1, fp16=True, beta=0.1,
            max_length=512, max_prompt_length=256,
            logging_steps=10, save_steps=999, report_to="none",
            remove_unused_columns=False,
        )
        trainer = DPOTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)
        trainer.train()
        print(f"Iteration {it} done.")

    model.save_pretrained(CKPT_SPIN)
    tok.save_pretrained(CKPT_SPIN)
    print(f"SPIN saved: {CKPT_SPIN}")

    # Eval post-SPIN (Scenario B data)
    model.eval()
    results["B"] = eval_model(model, tok, label="B — SPIN")
    with open(f"{RESULTS_DIR}/scenario_B.json", "w") as f:
        json.dump(results["B"], f, indent=2)
else:
    print("Skipping SPIN (not needed for this scenario).")

---
# Section 4 — H-AZR Training (Scenarios C and D)

In [ ]:
# 4.1 — H-Neuron penalty function
def compute_h_penalty(model, tok, text):
    acts = {}
    hooks = []
    try: layers = model.base_model.model.model.layers
    except: layers = model.model.layers
    for i, layer in enumerate(layers):
        def hook(m, inp, out, idx=i): acts[idx] = out.detach().cpu().float()
        hooks.append(layer.mlp.register_forward_hook(hook))
    inp = tok(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad(): model(**inp)
    for h in hooks: h.remove()
    pens = []
    for ls, info in h_neurons.items():
        li = int(ls)
        if li not in acts: continue
        a = acts[li][0, -1, :]
        idx = torch.tensor(info["indices"])
        pens.append(a[idx].abs().mean().item())
    return float(np.mean(pens)) if pens else 0.0

def hazr_reward(completions, prompts=None, **kwargs):
    rewards = []
    for comp in completions:
        code = comp
        if "```python" in code: code = code.split("```python")[1].split("```")[0]
        elif "```" in code: code = code.split("```")[1].split("```")[0]
        # Accuracy: does the code run?
        acc = 1.0 if run_code(code, "# no test") else 0.0
        # H-Neuron penalty
        pen = compute_h_penalty(model, tok, comp)
        rewards.append(acc - LAMBDA_H * pen)
    return rewards

print("H-AZR reward ready.")

In [ ]:
# 4.2 — H-AZR training
from trl import GRPOConfig, GRPOTrainer

# Load from SPIN checkpoint (Scenario C) or base model (Scenario D)
del model; gc.collect(); torch.cuda.empty_cache()

base = CKPT_SPIN if (SCENARIO == "C" and os.path.exists(CKPT_SPIN)) else None
model, tok = load_model(base)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj",
                                  "gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)

TRAIN_PROMPTS = [
    "Write a Python function that reverses a string.",
    "Write a Python function that checks if a number is prime.",
    "Write a Python function that computes factorial recursively.",
    "Write a Python function that finds the max in a list.",
    "Write a Python function that merges two sorted lists.",
    "Write a Python function that detects if a string is an anagram.",
    "Write a Python function that counts word frequency.",
    "Write a Python function that implements a stack.",
] * 16

train_ds = Dataset.from_dict({"prompt": TRAIN_PROMPTS})

grpo_cfg = GRPOConfig(
    output_dir=CKPT_HAZR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_ratio=0.05,
    fp16=True,
    num_generations=4,
    max_new_tokens=256,
    temperature=0.8,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model, args=grpo_cfg,
    train_dataset=train_ds,
    reward_funcs=[hazr_reward],
    processing_class=tok,
)

print(f"Starting H-AZR training — Scenario {SCENARIO}")
trainer.train()
model.save_pretrained(CKPT_HAZR)
tok.save_pretrained(CKPT_HAZR)
print(f"Saved: {CKPT_HAZR}")

In [ ]:
# 4.3 — Eval post H-AZR
model.eval()
results[SCENARIO] = eval_model(model, tok, label=f"{SCENARIO} — H-AZR")
with open(f"{RESULTS_DIR}/scenario_{SCENARIO}.json", "w") as f:
    json.dump(results[SCENARIO], f, indent=2)
print("Saved.")

---
# Section 5 — Final Table (Paper Table 1)

In [ ]:
# 5.1 — Load all saved results and print Table 1
labels = {"A": "Baseline", "B": "SPIN only", "C": "Full H-AZR", "D": "H-AZR no warmup"}
all_results = {}

for sc in ["A", "B", "C", "D"]:
    path = f"{RESULTS_DIR}/scenario_{sc}.json"
    if os.path.exists(path):
        with open(path) as f:
            all_results[sc] = json.load(f)

print(f"\n{'='*62}")
print(f"  KAIL H-AZR — Table 1")
print(f"{'='*62}")
print(f"  {'Scenario':<22} {'TruthfulQA':>11} {'Hall.Rate':>10} {'Code':>8}")
print(f"  {'-'*56}")
for sc, r in all_results.items():
    name = f"{sc} — {labels[sc]}"
    print(f"  {name:<22} {r['truthfulqa_accuracy']:>11.3f} {r['hallucination_rate']:>10.3f} {r['code_pass_rate']:>8.3f}")
print(f"{'='*62}")